# 01 — Synthetic Data Generation

Generates the Riverside Health System synthetic denial datasets used throughout this portfolio.

**Important:** Claim-level data is **synthetic**, calibrated to CMS Medicare Physician & Other Practitioners PUF 2024 (Wisconsin) allowed-amount benchmarks.

Outputs written to `data/processed/`:
- `quarterly_claims_summary.csv` — population quarterly volumes/rates (8.2% / 9.6% / 11.7%)
- `riverside_denial_log.csv` — 500 claim-level denial records
- `analysis_constants.csv` — shared constants including headline revenue exposure


In [1]:
from pathlib import Path
import sys

# Resolve repo root from notebooks/ folder
ROOT = Path.cwd().resolve()
if (ROOT / 'data' / 'processed').exists():
    pass
elif (ROOT.parent.parent / 'data' / 'processed').exists():
    ROOT = ROOT.parent.parent
elif (ROOT.parent / 'data' / 'processed').exists():
    ROOT = ROOT.parent
else:
    raise FileNotFoundError('Could not locate repo root with data/processed')

sys.path.insert(0, str(ROOT / '02_denial_analysis' / 'src'))
print('ROOT =', ROOT)


ROOT = /workspace


In [2]:
import generate_synthetic_data as gen

out = ROOT / 'data' / 'processed'
# Call module main pieces directly for transparency
import numpy as np
rng = np.random.default_rng(gen.SEED)
quarterly = gen.build_quarterly_summary()
constants = gen.build_constants(quarterly)
denial_log = gen.build_denial_log(rng)
out.mkdir(parents=True, exist_ok=True)
quarterly.to_csv(out / 'quarterly_claims_summary.csv', index=False)
constants.to_csv(out / 'analysis_constants.csv', index=False)
denial_log.to_csv(out / 'riverside_denial_log.csv', index=False)
print('Wrote', len(denial_log), 'denial rows to', out)
quarterly


Wrote 500 denial rows to /workspace/data/processed


,Quarter,Claims_Submitted,Claims_Denied,Denial_Rate,Denial_Rate_Pct,Avg_Allowed_Amount,Revenue_at_Risk
0,Q3 2024,168000,13776,0.082,8.2,79.73,1098360.48
1,Q4 2024,172000,16512,0.096,9.6,79.73,1316501.76
2,Q1 2025,171000,20007,0.117,11.7,79.73,1595158.11


In [3]:
constants


,Constant,Value
0,AVG_ALLOWED_AMOUNT,79.730
1,ANNUAL_CLAIMS,680000.000
2,BASELINE_DENIAL_RATE,0.082
3,LATEST_DENIAL_RATE,0.117
4,BASELINE_ANNUAL_REVENUE_AT_RISK,4445744.800
5,LATEST_ANNUAL_REVENUE_AT_RISK,6343318.800
6,INCREMENTAL_ANNUAL_REVENUE_EXPOSURE,1897574.000
7,DENIAL_LOG_ROW_COUNT,500.000
8,RANDOM_SEED,42.000


In [4]:
denial_log['Denial_Code'].value_counts(normalize=True).mul(100).round(1)


Denial_Code
CO-4     28.0
CO-16    24.0
PR-96    19.0
CO-29     8.0
CO-50     7.0
CO-97     6.0
PR-1      5.0
OA-23     3.0
Name: proportion, dtype: float64